# Nemotron SFT Smoke Run — v1.1 traces — no TRL dependency

Purpose:
- consume `train_traces_v1_1.jsonl` from Notebook A
- run a **tiny manual PEFT/LoRA smoke update**
- avoid `trl` / `SFTTrainer` because the pinned Kaggle environment may not include TRL and internet is off
- prove adapter training step + adapter save path

This is **not** leaderboard optimization and **not** final submission packaging.


In [1]:
import gc
import json
import os
import random
import re
import shutil
import site
import stat
import sys
import zipfile
from pathlib import Path

import pandas as pd

TRACE_VERSION = "v1_1"
SMOKE_SAMPLE_SIZE = 6       # intentionally tiny; increase only after backward + save works
RANDOM_SEED = 42
MAX_SEQ_LEN = 384           # keep short to reduce activation memory
LR = 2e-4

LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

WORKING_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORKING_DIR / "adapter_smoke_v1_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRACE_JSONL_CANDIDATES = [WORKING_DIR / "train_traces_v1_1.jsonl"]
if Path("/kaggle/input").exists():
    TRACE_JSONL_CANDIDATES.extend(Path("/kaggle/input").rglob("train_traces_v1_1.jsonl"))

TRACE_JSONL_PATH = next((p for p in TRACE_JSONL_CANDIDATES if p.exists()), None)
if TRACE_JSONL_PATH is None:
    print("Known trace candidates:")
    for p in TRACE_JSONL_CANDIDATES[:50]:
        print(" -", p)
    raise FileNotFoundError("Could not find train_traces_v1_1.jsonl. Run Notebook A or add its output as input.")

print("TRACE_JSONL_PATH:", TRACE_JSONL_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


TRACE_JSONL_PATH: /kaggle/input/notebooks/jatalepawan/nemotron-trace-generator-v1-2026-04-29/train_traces_v1_1.jsonl
OUTPUT_DIR: /kaggle/working/adapter_smoke_v1_1


## Add NVIDIA utility/package paths

Nemotron requires `mamba_ssm`. In the successful submission demo, it comes from the Ryan Holbrook NVIDIA utility script. Add that utility script input to this Kaggle notebook if `mamba_ssm` still fails to import.


In [2]:
UTILITY_ROOT_CANDIDATES = [
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script"),
    Path("/kaggle/input/nvidia-utility-script"),
    Path("/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script"),
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script"),
]

for root in UTILITY_ROOT_CANDIDATES:
    if not root.exists():
        continue
    for sub in [root, root / "nvidia_cutlass_dsl/python_packages", root / "python_packages"]:
        if sub.exists():
            site.addsitedir(str(sub))
            if str(sub) not in sys.path:
                sys.path.insert(0, str(sub))
            print("Added package path:", sub)

print("Utility roots found:", [str(p) for p in UTILITY_ROOT_CANDIDATES if p.exists()])


Added package path: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script
Added package path: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages
Utility roots found: ['/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script']


## Imports

This version deliberately avoids `trl` and `datasets`. It uses a minimal manual training loop with PEFT.


In [3]:
import torch
import torch.nn.functional as F

import kagglehub
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

try:
    import mamba_ssm
    print("mamba_ssm import OK:", getattr(mamba_ssm, "__file__", "unknown"))
except Exception as e:
    print("mamba_ssm import failed. Utility roots:")
    for p in UTILITY_ROOT_CANDIDATES:
        print(" -", p, "exists=", p.exists())
    raise ImportError(
        "mamba_ssm is missing. Add the Kaggle input/notebook `ryanholbrook/nvidia-utility-script` "
        "or run this notebook in the same environment as the successful submission demo."
    ) from e

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


mamba_ssm import OK: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/mamba_ssm/__init__.py
torch: 2.12.0.dev20260324+cu128
cuda available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Runtime patches for Nemotron/Mamba/Triton robustness

In [4]:
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        try:
            mod.rmsnorm_fn = _pure_rmsnorm_fn
        except Exception:
            pass

PTXAS_BLACKWELL_CANDIDATES = [
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"),
    Path("/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script/triton/backends/nvidia/bin/ptxas-blackwell"),
]
ptxas_src = next((p for p in PTXAS_BLACKWELL_CANDIDATES if p.exists()), None)
if ptxas_src is not None:
    dst = Path("/tmp/ptxas-blackwell")
    shutil.copy2(ptxas_src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = str(dst)
    print("Copied ptxas-blackwell to:", dst)
else:
    print("ptxas-blackwell utility binary not found; continuing.")

try:
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
    print("Patched Triton get_ptxas_version.")
except Exception as e:
    print("Could not patch Triton compiler version:", repr(e))


ptxas-blackwell utility binary not found; continuing.
Patched Triton get_ptxas_version.


## Load v1.1 trace corpus and build tiny stratified smoke sample

In [5]:
records = []
with open(TRACE_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

assert records, "No records found."
df = pd.DataFrame(records)
print("Total trace records:", len(df))
print(df[["id", "category", "answer", "approx_trace_chars"]].head())
print("\nCategory counts:")
print(df["category"].value_counts())

per_category = max(1, SMOKE_SAMPLE_SIZE // df["category"].nunique())
sampled = (
    df.groupby("category", group_keys=False)
      .apply(lambda x: x.sample(min(len(x), per_category), random_state=RANDOM_SEED))
      .sample(frac=1.0, random_state=RANDOM_SEED)
      .head(SMOKE_SAMPLE_SIZE)
      .reset_index(drop=True)
)

print("\nSmoke sample shape:", sampled.shape)
print(sampled["category"].value_counts())


Total trace records: 4767
         id         category   answer  approx_trace_chars
0  001b24c4          numeral  XXXVIII                 178
1  00208201  unit_conversion    16.65                 302
2  0040ff76          gravity   154.62                 218
3  00463d04          gravity    50.51                 216
4  0047365c  unit_conversion    10.62                 299

Category counts:
category
gravity            1597
unit_conversion    1594
numeral            1576
Name: count, dtype: int64

Smoke sample shape: (6, 6)
category
gravity            2
unit_conversion    2
numeral            2
Name: count, dtype: int64


/tmp/ipykernel_65/3382150590.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), per_category), random_state=RANDOM_SEED))


In [6]:
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print("MODEL_PATH:", MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def build_training_text(row):
    messages = row["messages"]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        user = messages[0]["content"]
        assistant = messages[1]["content"]
        return f"User:\n{user}\n\nAssistant:\n{assistant}{tokenizer.eos_token}"

sampled["text"] = sampled.apply(build_training_text, axis=1)
print("Example text length:", len(sampled.loc[0, "text"]))
print(sampled.loc[0, "text"][:1200])


MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Example text length: 684
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, the gravitational constant has been secretly changed. Here are some example observations:
For t = 1.04s, distance = 3.93 m
For t = 2.21s, distance = 17.76 m
For t = 1.75s, distance = 11.14 m
Now, determine the falling distance for t = 4.19s given d = 0.5*g*t^2.
Please put your final answer inside \boxed{}. For example: \boxed{your answer}<|im_end|>
<|im_start|>assistant
<think></think>Step 1: Use d = 0.5 * g * t^2.
Step 2: Estimate g from the examples.
Step 3: The inferred g is approximately 7.2716.
Step 4: Substitute t = 4.19.
Step 5: The final numeric value is 63.83.
Final answer: \boxed{63.85}<|im_end|>



## Load model and attach LoRA

This follows the successful submission demo more closely: `device_map=auto`, `dtype=bfloat16`, and LoRA target regex for Nemotron projection modules.


In [7]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="auto",
)

for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
        try:
            mod.is_fast_path_available = False
            print(f"Patched {name}: is_fast_path_available = False")
        except Exception:
            pass

model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.train()


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): NemotronHForCausalLM(
      (backbone): NemotronHModel(
        (embeddings): Embedding(131072, 2688)
        (layers): ModuleList(
          (0): NemotronHBlock(
            (norm): NemotronHRMSNorm()
            (mixer): NemotronHMamba2Mixer(
              (act): SiLUActivation()
              (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144)
              (in_proj): lora.Linear(
                (base_layer): Linear(in_features=2688, out_features=10304, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2688, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=10304, bias=False)
                )
                (lora_embedd

## Tiny manual SFT update

This trains only a few examples. The goal is backward + optimizer + adapter save, not score.


In [8]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
assert trainable_params, "No trainable LoRA parameters found."

optimizer = torch.optim.AdamW(trainable_params, lr=LR)
losses = []

for step, text in enumerate(sampled["text"].tolist(), start=1):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    first_device = next(model.parameters()).device
    enc = {k: v.to(first_device) for k, v in enc.items()}
    labels = enc["input_ids"].clone()

    optimizer.zero_grad(set_to_none=True)
    out = model(**enc, labels=labels)
    loss = out.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()

    loss_value = float(loss.detach().cpu())
    losses.append(loss_value)
    print(f"step {step:02d}/{len(sampled)} | loss={loss_value:.4f}")

print("Manual SFT smoke loop complete.")
print("mean_loss:", sum(losses) / len(losses))


step 01/6 | loss=4.6033
step 02/6 | loss=3.3592
step 03/6 | loss=3.9880
step 04/6 | loss=3.9639
step 05/6 | loss=2.6959
step 06/6 | loss=2.7336
Manual SFT smoke loop complete.
mean_loss: 3.5573195616404214


## Save adapter artifacts and validate folder

In [9]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer_debug")

print("Adapter output files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUTPUT_DIR), f"({p.stat().st_size/1024:.1f} KB)")

adapter_config = OUTPUT_DIR / "adapter_config.json"
adapter_safetensors = OUTPUT_DIR / "adapter_model.safetensors"
adapter_bin = OUTPUT_DIR / "adapter_model.bin"

assert adapter_config.exists(), "Missing adapter_config.json"
assert adapter_safetensors.exists() or adapter_bin.exists(), "Missing adapter model weights"

print("\nSmoke adapter save validation passed.")
print(OUTPUT_DIR)


Adapter output files:
 - README.md (5.2 KB)
 - adapter_config.json (1.0 KB)
 - adapter_model.safetensors (3439795.2 KB)
 - tokenizer_debug/chat_template.jinja (10.3 KB)
 - tokenizer_debug/tokenizer.json (16677.3 KB)
 - tokenizer_debug/tokenizer_config.json (0.4 KB)

Smoke adapter save validation passed.
/kaggle/working/adapter_smoke_v1_1


## Optional debug zip

This zip is for moving the smoke adapter to a validation notebook. It is **not** final submission packaging.


In [10]:
SMOKE_ZIP_PATH = WORKING_DIR / "adapter_smoke_v1_1.zip"

with zipfile.ZipFile(SMOKE_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.iterdir():
        if p.is_file() and p.name.startswith("adapter_"):
            zf.write(p, arcname=p.name)

print("Created:", SMOKE_ZIP_PATH, f"({SMOKE_ZIP_PATH.stat().st_size/1024/1024:.2f} MB)")
with zipfile.ZipFile(SMOKE_ZIP_PATH, "r") as zf:
    print("Zip contents:", zf.namelist())
    assert "adapter_config.json" in zf.namelist()


Created: /kaggle/working/adapter_smoke_v1_1.zip (2425.52 MB)
Zip contents: ['adapter_config.json', 'adapter_model.safetensors']
